# 🎬 Text-to-Video: Human Motion in Dynamic Environments
## Phase 1: Foundation & Analysis (Refactored Pipeline)

**Theme:** Human motion in dynamic environments (people walking, crowded streets, human interactions).
**Objective:** Advanced analysis including real CLIP-SIM, SSIM temporal analysis, structured experiments, and weakness heuristics.

## 1. Install Dependencies

In [1]:
!pip install -q torch diffusers transformers accelerate imageio[ffmpeg] opencv-python pillow scikit-image numpy

## 2. Refactored Generation & Evaluation Pipeline

In [2]:
import os
import time
import json
import logging
import torch
import numpy as np
import imageio
from typing import List, Dict, Any
from diffusers import StableDiffusionXLPipeline, StableVideoDiffusionPipeline
from diffusers.utils import export_to_video
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from skimage.metrics import structural_similarity as ssim_fn

# Configure structured logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# --- Evaluation Metrics ---

def calculate_clip_sim(frames: List[Image.Image], prompt: str, device: str = "cuda") -> float:
    """Real implementation of CLIP-SIM (Semantic Alignment) across frames."""
    logger.info("Calculating CLIP-SIM across generated frames...")
    try:
        model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
        processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
        
        inputs = processor(text=[prompt], images=frames, return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            # logits_per_image represents the image-text similarity score
            # Shape: [num_frames, num_texts(1)]
            similarity_scores = outputs.logits_per_image.squeeze().cpu().numpy()
            
        # Clear VRAM
        del model, processor, inputs, outputs
        torch.cuda.empty_cache()
        return float(np.mean(similarity_scores))
    except Exception as e:
        logger.error(f"CLIP-SIM failed: {e}")
        return 0.0

def calculate_fvd(frames: List[Image.Image]) -> float:
    """Placeholder for Frechet Video Distance (FVD for Realism/Quality)."""
    logger.info("Calculating Frechet Video Distance (FVD)... [Placeholder]")
    return -1.0 # Requires feature extraction against a real video dataset

def calculate_temporal_metrics(frames: List[Image.Image]) -> Dict[str, float]:
    """Calculates frame differences and SSIM for Temporal Consistency."""
    logger.info("Calculating Temporal Consistency metrics (SSIM & Frame Diff)...")
    frames_np = [np.array(f) for f in frames]
    
    ssim_scores = []
    frame_diffs = []
    
    for i in range(len(frames_np) - 1):
        f1 = frames_np[i]
        f2 = frames_np[i+1]
        
        # Mean absolute pixel difference
        diff = np.mean(np.abs(f1.astype(np.float32) - f2.astype(np.float32)))
        frame_diffs.append(diff)
        
        # SSIM
        s = ssim_fn(f1, f2, channel_axis=2, data_range=255)
        ssim_scores.append(s)
        
    return {
        "mean_ssim": float(np.mean(ssim_scores)),
        "mean_frame_diff": float(np.mean(frame_diffs)),
        "ssim_variance": float(np.var(ssim_scores))
    }

def analyze_weakness_heuristics(metrics: Dict[str, float]) -> Dict[str, str]:
    """Heuristic logic to flag observations automatically."""
    analysis = {}
    
    # Temporal Consistency Heuristics
    mean_ssim = metrics.get("mean_ssim", 0)
    ssim_var = metrics.get("ssim_variance", 0)
    
    if mean_ssim > 0.85 and ssim_var < 0.01:
        analysis["temporal_consistency"] = "good"
        analysis["flickering_detected"] = "no"
    elif mean_ssim > 0.65:
        analysis["temporal_consistency"] = "moderate"
        analysis["flickering_detected"] = "minor"
    else:
        analysis["temporal_consistency"] = "poor"
        analysis["flickering_detected"] = "yes"
        
    # Semantic Alignment Heuristics
    clip_sim = metrics.get("clip_sim", 0)
    if clip_sim > 28.0:
        analysis["semantic_alignment"] = "good"
    elif clip_sim > 24.0:
        analysis["semantic_alignment"] = "moderate"
    else:
        analysis["semantic_alignment"] = "poor"
        
    analysis["motion_realism"] = "requires_manual_review_or_fvd"
    
    return analysis

# --- Model Pipeline ---

def load_models():
    """Loads Text-to-Image (SDXL) and Image-to-Video (SVD) models."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    logger.info(f"Initializing models on hardware: {device}")

    logger.info("Loading SDXL Base 1.0 (Text-to-Image)...")
    t2i_pipe = StableDiffusionXLPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0",
        torch_dtype=torch.float16,
        variant="fp16",
        use_safetensors=True
    )
    if device == "cuda":
        t2i_pipe.enable_model_cpu_offload()

    logger.info("Loading Stable Video Diffusion XT (Image-to-Video)...")
    i2v_pipe = StableVideoDiffusionPipeline.from_pretrained(
        "stabilityai/stable-video-diffusion-img2vid-xt",
        torch_dtype=torch.float16,
        variant="fp16"
    )
    if device == "cuda":
        i2v_pipe.enable_model_cpu_offload()

    return t2i_pipe, i2v_pipe, device

def save_outputs(frames: List[Image.Image], output_dir: str, fps: int = 7) -> str:
    """Saves individual intermediate frames and compiles them into an MP4 video."""
    os.makedirs(output_dir, exist_ok=True)
    
    frames_dir = os.path.join(output_dir, "frames")
    os.makedirs(frames_dir, exist_ok=True)
    for i, frame in enumerate(frames):
        frame.save(os.path.join(frames_dir, f"frame_{i:03d}.png"))
    
    video_path = os.path.join(output_dir, "output_video.mp4")
    export_to_video(frames, video_path, fps=fps)
    return video_path

def generate_video_from_text(
    t2i_pipe, i2v_pipe, device: str,
    prompt: str,
    complexity_level: str,
    output_dir: str,
    num_inference_steps_t2i: int = 30,
    guidance_scale_t2i: float = 7.5,
    num_inference_steps_i2v: int = 25,
    motion_bucket_id: int = 127,
    noise_aug_strength: float = 0.1,
    seed: int = 42
) -> Dict[str, Any]:
    
    start_time = time.time()
    generator = torch.Generator(device="cpu").manual_seed(seed)
    
    logger.info(f"Generating base image for prompt -> '{prompt[:40]}...'")
    initial_image = t2i_pipe(
        prompt=prompt,
        num_inference_steps=num_inference_steps_t2i,
        guidance_scale=guidance_scale_t2i,
        height=576, width=1024,
        generator=generator
    ).images[0]

    logger.info("Animating image into video sequence...")
    frames = i2v_pipe(
        initial_image,
        decode_chunk_size=8,
        generator=generator,
        motion_bucket_id=motion_bucket_id,
        noise_aug_strength=noise_aug_strength,
        num_inference_steps=num_inference_steps_i2v
    ).frames[0]

    logger.info("Saving visual outputs...")
    video_path = save_outputs(frames, output_dir, fps=7)
    
    # Calculate Metrics
    temporal_metrics = calculate_temporal_metrics(frames)
    clip_sim = calculate_clip_sim(frames, prompt, device)
    fvd = calculate_fvd(frames)
    
    raw_metrics = {
        "clip_sim": clip_sim,
        "fvd": fvd,
        "mean_ssim": temporal_metrics["mean_ssim"],
        "mean_frame_diff": temporal_metrics["mean_frame_diff"],
        "ssim_variance": temporal_metrics["ssim_variance"]
    }
    
    # Weakness Analysis
    weakness_observations = analyze_weakness_heuristics(raw_metrics)
    
    generation_time = time.time() - start_time

    metadata = {
        "prompt": prompt,
        "complexity_level": complexity_level,
        "seed": seed,
        "parameters": {
            "num_inference_steps_t2i": num_inference_steps_t2i,
            "guidance_scale_t2i": guidance_scale_t2i,
            "num_inference_steps_i2v": num_inference_steps_i2v,
            "motion_bucket_id": motion_bucket_id,
            "noise_aug_strength": noise_aug_strength
        },
        "generation_time_seconds": round(generation_time, 2),
        "video_path": video_path,
        "metrics": raw_metrics,
        "weakness_analysis": weakness_observations
    }
    
    with open(os.path.join(output_dir, "metadata.json"), "w") as f:
        json.dump(metadata, f, indent=4)
        
    return metadata

# --- Experiment Runner ---

def run_structured_experiments(base_output_path: str = "/kaggle/working/experiment_results"):
    t2i_pipe, i2v_pipe, device = load_models()
    
    prompts = {
        "simple": "A person walking on an empty sunlit street, smooth motion, highly detailed, cinematic lighting, photorealistic",
        "medium": "A person walking in a busy street with moving cars and background elements, daytime, dynamic motion, cinematic tracking shot",
        "complex": "A crowded public market with many diverse people walking and interacting, merchants selling goods, vibrant colors, chaotic but realistic human motion, cinematic wide camera"
    }
    
    all_results = []
    
    # EXPERIMENT A: Same prompt (medium), Different Guidance Scales
    logger.info("Starting Experiment A: Guidance Scale Variations")
    exp_a_path = os.path.join(base_output_path, "Exp_A_Guidance")
    for gs in [5.0, 7.5, 10.0]:
        out_dir = os.path.join(exp_a_path, f"guidance_{str(gs).replace('.','_')}")
        logger.info(f"Running Exp A -> GS: {gs}")
        res = generate_video_from_text(
            t2i_pipe, i2v_pipe, device,
            prompt=prompts["medium"], complexity_level="medium",
            output_dir=out_dir, guidance_scale_t2i=gs, seed=1024
        )
        res["experiment_type"] = f"Guidance_Scale_{gs}"
        all_results.append(res)
        
    # EXPERIMENT B: Same prompt (medium), Different Inference Steps (I2V)
    logger.info("Starting Experiment B: Inference Steps Variations")
    exp_b_path = os.path.join(base_output_path, "Exp_B_Steps")
    for steps in [15, 25, 40]:
        out_dir = os.path.join(exp_b_path, f"steps_{steps}")
        logger.info(f"Running Exp B -> Steps: {steps}")
        res = generate_video_from_text(
            t2i_pipe, i2v_pipe, device,
            prompt=prompts["medium"], complexity_level="medium",
            output_dir=out_dir, num_inference_steps_i2v=steps, seed=1024
        )
        res["experiment_type"] = f"Inference_Steps_{steps}"
        all_results.append(res)
        
    # EXPERIMENT C: Compare across complexity levels
    logger.info("Starting Experiment C: Complexity Levels")
    exp_c_path = os.path.join(base_output_path, "Exp_C_Complexity")
    for level, p_text in prompts.items():
        out_dir = os.path.join(exp_c_path, f"complexity_{level}")
        logger.info(f"Running Exp C -> Level: {level}")
        res = generate_video_from_text(
            t2i_pipe, i2v_pipe, device,
            prompt=p_text, complexity_level=level,
            output_dir=out_dir, seed=2048
        )
        res["experiment_type"] = f"Complexity_{level}"
        all_results.append(res)

    # Save Global Summary
    summary_path = os.path.join(base_output_path, "experiment_summary.json")
    with open(summary_path, "w") as f:
        json.dump(all_results, f, indent=4)
        
    logger.info(f"All experiments finished! Global summary saved to {summary_path}")

# Run everything
run_structured_experiments()

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

...

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 3. Review Experiment Results & Weakness Analysis

In [3]:
import pandas as pd
import json

try:
    with open('/kaggle/working/experiment_results/experiment_summary.json', 'r') as f:
        results = json.load(f)
        
    df = pd.json_normalize(results)
    
    # Select important columns for the report
    display_cols = [
        'experiment_type', 'complexity_level', 
        'metrics.clip_sim', 'metrics.mean_ssim', 
        'weakness_analysis.temporal_consistency',
        'weakness_analysis.semantic_alignment'
    ]
    
    print("--- GLOBAL EXPERIMENT REPORT ---")
    display(df[display_cols].sort_values(by="experiment_type"))
    
except FileNotFoundError:
    print("Experiment hasn't completed yet.")

--- GLOBAL EXPERIMENT REPORT ---


,experiment_type,complexity_level,metrics.clip_sim,metrics.mean_ssim,weakness_analysis.temporal_consistency,weakness_analysis.semantic_alignment
8,Complexity_complex,complex,31.943998,0.590045,poor,good
7,Complexity_medium,medium,29.170944,0.698240,moderate,good
6,Complexity_simple,simple,33.554028,0.802451,moderate,good
2,Guidance_Scale_10.0,medium,32.292274,0.781989,moderate,good
0,Guidance_Scale_5.0,medium,32.751659,0.774977,moderate,good
1,Guidance_Scale_7.5,medium,32.009647,0.760021,moderate,good
3,Inference_Steps_15,medium,32.247574,0.774673,moderate,good
4,Inference_Steps_25,medium,32.009647,0.760021,moderate,good
5,Inference_Steps_40,medium,32.068134,0.756174,moderate,good


In [4]:
!zip -r results.zip /kaggle/working/experiment_results/


  adding: kaggle/working/experiment_results/ (stored 0%)
  adding: kaggle/working/experiment_results/experiment_summary.json (deflated 87%)
  adding: kaggle/working/experiment_results/Exp_A_Guidance/ (stored 0%)
  adding: kaggle/working/experiment_results/Exp_A_Guidance/guidance_10_0/ (stored 0%)
  adding: kaggle/working/experiment_results/Exp_A_Guidance/guidance_10_0/frames/ (stored 0%)
  adding: kaggle/working/experiment_results/Exp_A_Guidance/guidance_10_0/frames/frame_016.png (deflated 0%)
  adding: kaggle/working/experiment_results/Exp_A_Guidance/guidance_10_0/frames/frame_021.png (deflated 0%)
  adding: kaggle/working/experiment_results/Exp_A_Guidance/guidance_10_0/frames/frame_007.png (deflated 0%)
  adding: kaggle/working/experiment_results/Exp_A_Guidance/guidance_10_0/frames/frame_020.png (deflated 0%)
  adding: kaggle/working/experiment_results/Exp_A_Guidance/guidance_10_0/frames/frame_006.png (deflated 0%)
  adding: kaggle/working/experiment_results/Exp_A_Guidance/guidance_1